# QUIBC — Training Demonstration

Trains QUIBC on the CLIC dataset under simulated IoT conditions.

**Paper**: *QUIBC: A Quantum-Inspired Image Binarization Compressor for Resource-Constrained Edge Devices* — IEEE INDICON 2025

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from quibc import QUIBC, make_encoder, make_decoder, build_clic_datasets
from quibc.losses import psnr_metric, ms_ssim_metric, entropy_bits
from quibc.train import exponential_decay_schedule, build_callbacks
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Hyperparameters

In [ ]:
IMG_SIZE  = 256
LATENT_CH = 96
BATCH     = 16
EPOCHS    = 38
LAMBDA    = 1e-3
SEED      = 42
tf.keras.utils.set_random_seed(SEED)

## 2. CLIC Dataset with IoT Augmentation

In [ ]:
train_ds, val_ds = build_clic_datasets(
    img_size=IMG_SIZE, batch_size=BATCH,
    seed=SEED, iot_augment=True, cache=True
)
print('Datasets ready.')

## 3. Preview Training Samples

In [ ]:
sample_imgs = next(iter(train_ds))[0][:4].numpy()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img in zip(axes, sample_imgs):
    ax.imshow(img); ax.axis('off')
fig.suptitle('CLIC Training Images (IoT-degraded)', fontsize=13)
plt.tight_layout(); plt.show()

## 4. Build QUIBC Architecture

In [ ]:
encoder = make_encoder(IMG_SIZE, LATENT_CH)
decoder = make_decoder(IMG_SIZE, LATENT_CH)
model   = QUIBC(encoder, decoder, lam=LAMBDA)
model.compile(optimizer=tf.keras.optimizers.Adam(exponential_decay_schedule(1e-4)))
_ = model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)), training=False)
encoder.summary()

## 5. Train

In [ ]:
history = model.fit(
    train_ds, epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=build_callbacks('../checkpoints')
)

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (k, lbl) in zip(axes.flat, [('loss','Loss'),('psnr','PSNR (dB)'),('ms_ssim','MS-SSIM'),('rate_bits','Rate (bits)')]):
    ax.plot(history.history[k], label='Train')
    ax.plot(history.history[f'val_{k}'], label='Val', linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel(lbl); ax.set_title(lbl); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('QUIBC Training Curves', fontsize=14)
plt.tight_layout(); plt.show()
print(f'Best val PSNR:    {max(history.history["val_psnr"]):.2f} dB')
print(f'Best val MS-SSIM: {max(history.history["val_ms_ssim"]):.4f}')
print(f'Effective lambda: {model.effective_lambda:.6f}')

## 7. Qualitative Reconstruction

In [ ]:
originals = next(iter(val_ds))[0][:3]
bits, probs = model.encoder(originals, training=False)
recons = model.decoder(bits, training=False)
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for col, t in enumerate(['Original','Reconstruction','Abs Difference']):
    axes[0,col].set_title(t, fontsize=12, fontweight='bold')
for row in range(3):
    o, r = originals[row].numpy(), recons[row].numpy()
    p = float(tf.image.psnr(originals[row:row+1], recons[row:row+1], 1.0).numpy())
    axes[row,0].imshow(o); axes[row,0].axis('off')
    axes[row,1].imshow(r); axes[row,1].set_title(f'PSNR={p:.2f} dB', fontsize=10); axes[row,1].axis('off')
    axes[row,2].imshow(np.abs(o-r), cmap='hot'); axes[row,2].axis('off')
plt.suptitle('QUIBC Reconstruction Quality', fontsize=14)
plt.tight_layout(); plt.show()

## 8. Rate–Distortion Scatter

In [ ]:
psnrs, bpps = [], []
for i, (x, _) in enumerate(val_ds):
    if i >= 20: break
    bits, probs = model.encoder(x, training=False)
    recon = model.decoder(bits, training=False)
    psnrs.extend(psnr_metric(x, recon).numpy())
    bpp = float((entropy_bits(probs)*(IMG_SIZE//8)**2/IMG_SIZE**2).numpy())
    bpps.extend([bpp]*len(x))
plt.figure(figsize=(8,5))
plt.scatter(bpps, psnrs, alpha=0.6, s=20)
plt.xlabel('bpp'); plt.ylabel('PSNR (dB)')
plt.title('Rate–Distortion Scatter (CLIC Validation)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()